In [1]:
import pandas as pd
import matplotlib.pyplot as plt
# Use only Domestic/Men's/IPL/2026 data

In [2]:
data = pd.read_csv(f'../data/Domestic/Men\'s/IPL/2026/ipl_2026_deliveries.csv')
data['date'] = pd.to_datetime(data['date'], errors='coerce')
data.info()
data.sample(frac=1).reset_index(drop=True).head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17477 entries, 0 to 17476
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   match_id          17477 non-null  int64         
 1   season            17477 non-null  int64         
 2   phase             17477 non-null  object        
 3   match_no          17477 non-null  int64         
 4   date              17477 non-null  datetime64[ns]
 5   venue             17477 non-null  object        
 6   batting_team      17477 non-null  object        
 7   bowling_team      17477 non-null  object        
 8   innings           17477 non-null  int64         
 9   over              17477 non-null  float64       
 10  striker           17477 non-null  object        
 11  bowler            17477 non-null  object        
 12  runs_of_bat       17477 non-null  int64         
 13  extras            17477 non-null  int64         
 14  wide              1747

,match_id,season,phase,match_no,date,venue,batting_team,bowling_team,innings,over,...,bowler,runs_of_bat,extras,wide,legbyes,byes,noballs,wicket_type,player_dismissed,fielder
0,202611,2026,Group Stage,11,2026-04-05,"M.Chinnaswamy Stadium, Bengaluru",CSK,RCB,2,8.2,...,Krunal Pandya,1,0,0,0,0,0,NaN,NaN,NaN
1,202641,2026,Group Stage,41,2026-04-29,"Wankhede Stadium, Mumbai",MI,SRH,1,16.2,...,Pat Cummins,1,0,0,0,0,0,NaN,NaN,NaN
2,202645,2026,Group Stage,45,2026-05-03,"Rajiv Gandhi International Stadium, Hyderabad",KKR,SRH,2,5.3,...,Eshan Malinga,1,0,0,0,0,0,NaN,NaN,NaN
3,202620,2026,Group Stage,20,2026-04-12,"Wankhede Stadium, Mumbai",RCB,MI,1,17.3,...,Hardik Pandya,1,0,0,0,0,0,NaN,NaN,NaN
4,202601,2026,Group Stage,1,2026-03-28,"M.Chinnaswamy Stadium, Bengaluru",SRH,RCB,1,8.4,...,Abhinandan Singh,4,0,0,0,0,0,NaN,NaN,NaN


In [3]:
# Further exploration of data
# Print unique values for each column
for column in data.columns:
    unique_values = data[column].unique()
    print(f"Column: {column}, Unique Values: {len(unique_values)}")
    if len(unique_values) <= 20:
        print(f"Values: {unique_values}")
    print("-" * 50)

unique_matches = data['match_id'].unique()
unique_teams = pd.concat([data['batting_team'], data['bowling_team']]).unique()
unique_players = pd.concat([data['striker'], data['bowler']]).unique()

Column: match_id, Unique Values: 74
--------------------------------------------------
Column: season, Unique Values: 1
Values: [2026]
--------------------------------------------------
Column: phase, Unique Values: 5
Values: ['Group Stage' 'Qualifier 1' 'Eliminator' 'Qualifier 2' 'Final']
--------------------------------------------------
Column: match_no, Unique Values: 74
--------------------------------------------------
Column: date, Unique Values: 62
--------------------------------------------------
Column: venue, Unique Values: 13
Values: ['M.Chinnaswamy Stadium, Bengaluru' 'Wankhede Stadium, Mumbai'
 'Barsapara Cricket Stadium, Guwahati'
 'Maharaja Yadavindra Singh International Cricket Stadium, Mullanpur, New Chandigarh'
 'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow'
 'Eden Gardens, Kolkata' 'MA Chidambaram Stadium, Chennai'
 'Arun Jaitley Stadium, Delhi' 'Narendra Modi Stadium, Ahmedabad'
 'Rajiv Gandhi International Stadium, Hyderabad'
 'Sawai Mans

In [4]:
def fantasy_points(deliveries, player):
    if 'match_id' in deliveries.columns:
        # We possible have multiple match data, return fantasy points for that match
        unique_matches = deliveries['match_id'].unique()
        if len(unique_matches) == 0:
            return 0
        if len(unique_matches) > 1:  # More than one match data, return average of each match's fantasy points
            return deliveries.groupby('match_id').apply(lambda x: fantasy_points(x, player), include_groups=False).mean()

    # Ensure deliveries are in chronological order
    deliveries = deliveries.sort_values(by=['innings', 'over'])
    f = 4  # Lineup points
    score = 0
    wickets = 0
    catches = 0
    over_score = 0
    for d in deliveries.itertuples():
        over_score += d.runs_of_bat
        if d.striker == player:
            f += d.runs_of_bat + (1 if d.runs_of_bat == 4 else 0) + (2 if d.runs_of_bat == 6 else 0)
            score += d.runs_of_bat
            if d.player_dismissed == player and score == 0:
                # TODO: no points are deducted if player is a bowler!!!
                f -= 2  # Deduct points for getting out for a duck
        if d.bowler == player:
            if not pd.isna(d.player_dismissed) and d.wicket_type not in ['runout', 'obstructing the field', 'retired out']:
                f += 25
                wickets += 1
        if d.fielder == player:
            f += (8 if d.wicket_type == 'caught' else 0) + (12 if d.wicket_type in ['runout', 'stumped'] else 0)
            catches += 1 if d.wicket_type == 'caught' else 0
        # Reset over
        if str(d.over).split('.')[1] == '6':
            if over_score == 0 and d.bowler == player:
                f += 12  # Add points for maiden over
            over_score = 0  # Reset score for the new over
    if wickets >= 3:
        f += 4 * (min(wickets, 5) - 2)  # Bonus points for taking 3-5 wickets: 4, 8, 12 points for 3, 4, 5 wickets respectively
    if score >= 50:
        f += 8
    if score >= 100:
        f += 16
    if catches >= 3:
        f += 4
    return f

In [5]:
# Temporary df handling running averages
temp_df = []
# Get match dates in ascending order
for m in unique_matches:
    # get players that have played in this match
    date = data[data['match_id'] == m]['date'].iloc[0]
    players = data[data['match_id'] == m][['striker', 'bowler']].melt(value_name='player')['player'].unique()
    for player in players:
        fp = fantasy_points(data[data['match_id'] == m], player)

        # Calculate total stats across all matches played by the player up to this match
        num_matches = data[(data['date'] < date) & ((data['striker'] == player) | (data['bowler'] == player) | (data['fielder'] == player))]['match_id'].nunique()
        total_score = data[(data['date'] < date) & (data['striker'] == player)]['runs_of_bat'].sum()
        total_balls_faced = data[(data['date'] < date) & (data['striker'] == player)].shape[0]
        total_outs = data[(data['date'] < date) & (data['striker'] == player)]['player_dismissed'].notna().sum()
        total_fours = data[(data['date'] < date) & (data['striker'] == player) & (data['runs_of_bat'] == 4)].shape[0]
        total_sixes = data[(data['date'] < date) & (data['striker'] == player) & (data['runs_of_bat'] == 6)].shape[0]
        total_balls_bowled = data[(data['date'] < date) & (data['bowler'] == player)].shape[0]
        total_runs_conceded = data[(data['date'] < date) & (data['bowler'] == player)]['runs_of_bat'].sum() + data[(data['date'] < date) & (data['bowler'] == player)]['extras'].sum()
        total_wickets_taken = data[(data['date'] < date) & (data['bowler'] == player)]['player_dismissed'].notna().sum()
        total_catches = data[(data['date'] < date) & (data['fielder'] == player) & (data['wicket_type'] == 'caught')].shape[0]
        total_run_outs = data[(data['date'] < date) & (data['fielder'] == player) & (data['wicket_type'] == 'runout')].shape[0]
        total_fantasy_points = fantasy_points(data[data['date'] < date], player)
        total_venue_fantasy_points = fantasy_points(data[(data['date'] < date) & (data['venue'] == data[data['match_id'] == m]['venue'].iloc[0])], player)
        total_against_opposition_fantasy_points = fantasy_points(data[(data['date'] < date) & ((data['batting_team'] == data[data['match_id'] == m]['bowling_team'].iloc[0]) | (data['bowling_team'] == data[data['match_id'] == m]['batting_team'].iloc[0]))], player)

        # Calculate running average:
        avg_score = total_score / total_balls_faced if total_balls_faced > 0 else 0
        avg_wickets_taken = total_wickets_taken / num_matches if num_matches > 0 else 0
        avg_fantasy_points = total_fantasy_points / num_matches if num_matches > 0 else 0
        avg_venue_fantasy_points = total_venue_fantasy_points / num_matches if num_matches > 0 else 0
        avg_against_opposition_fantasy_points = total_against_opposition_fantasy_points / num_matches if num_matches > 0 else 0
        avg_balls_faced = total_balls_faced /  num_matches if num_matches > 0 else 0
        avg_outs = total_outs / num_matches if num_matches > 0 else 0
        avg_fours = total_fours / num_matches if num_matches > 0 else 0
        avg_sixes = total_sixes / num_matches if num_matches > 0 else 0
        # TODO: byes legbyes shouldnt count in total_runs_conceded
        avg_runs_conceded = total_runs_conceded / total_balls_bowled if total_balls_bowled > 0 else 0
        avg_balls_bowled = total_balls_bowled / num_matches if num_matches > 0 else 0
        avg_catches = total_catches / num_matches if num_matches > 0 else 0
        avg_run_outs = total_run_outs / num_matches if num_matches > 0 else 0

        # Get a past 5 running average for the player
        past_matches = data[(data['date'] < date) & ((data['striker'] == player) | (data['bowler'] == player) | (data['fielder'] == player))].sort_values(by='date', ascending=False)['match_id'].unique()
        if len(past_matches) > 5:
            # Get the last 5 matches
            past_matches = past_matches[:5]
        # Get all matches if less than 5
        total_score_5 = data[(data['match_id'].isin(past_matches)) & (data['striker'] == player)]['runs_of_bat'].sum()
        total_balls_faced_5 = data[(data['match_id'].isin(past_matches)) & (data['striker'] == player)].shape[0]
        total_balls_bowled_5 = data[(data['match_id'].isin(past_matches)) & (data['bowler'] == player)].shape[0]
        total_outs_5 = data[(data['match_id'].isin(past_matches)) & (data['striker'] == player)]['player_dismissed'].notna().sum()
        total_runs_conceded_5 = data[(data['match_id'].isin(past_matches)) & (data['bowler'] == player)]['runs_of_bat'].sum() + data[(data['match_id'].isin(past_matches)) & (data['bowler'] == player)]['extras'].sum()
        total_wickets_taken_5 = data[(data['match_id'].isin(past_matches)) & (data['bowler'] == player)]['player_dismissed'].notna().sum()

        avg_score_5 = total_score_5 / total_balls_faced_5 if total_balls_faced_5 > 0 else 0
        avg_balls_faced_5 = total_balls_faced_5 /  len(past_matches) if len(past_matches) > 0 else 0
        avg_outs_5 = total_outs_5 / len(past_matches) if len(past_matches) > 0 else 0
        avg_catches_5 = data[(data['match_id'].isin(past_matches)) & (data['fielder'] == player)]['wicket_type'].eq('caught').sum() / len(past_matches) if len(past_matches) > 0 else 0
        avg_runs_conceded_5 = total_runs_conceded_5 / total_balls_bowled_5 if total_balls_bowled_5 > 0 else 0
        avg_balls_bowled_5 = total_balls_bowled_5 / len(past_matches) if len(past_matches) > 0 else 0
        avg_wickets_taken_5 = total_wickets_taken_5 / len(past_matches) if len(past_matches) > 0 else 0
        avg_fantasy_points_5 = fantasy_points(data[data['match_id'].isin(past_matches)], player)

        temp_df.append({
            'player': player,
            'match_id': m,
            'date': date,
            'venue': data[data['match_id'] == m]['venue'].iloc[0],
            'batting_team': data[data['match_id'] == m]['batting_team'].iloc[0],
            'bowling_team': data[data['match_id'] == m]['bowling_team'].iloc[0],
            'avg_score': avg_score,
            'avg_balls_faced': avg_balls_faced,
            'avg_outs': avg_outs,
            'avg_runs_conceded': avg_runs_conceded,
            'avg_balls_bowled': avg_balls_bowled,
            'avg_wickets_taken': avg_wickets_taken,
            'avg_fantasy_points': avg_fantasy_points,
            'avg_venue_fantasy_points': avg_venue_fantasy_points,
            'avg_against_opposition_fantasy_points': avg_against_opposition_fantasy_points,
            'avg_score_5': avg_score_5,
            'avg_balls_faced_5': avg_balls_faced_5,
            'avg_outs_5': avg_outs_5,
            'avg_runs_conceded_5': avg_runs_conceded_5,
            'avg_balls_bowled_5': avg_balls_bowled_5,
            'avg_wickets_taken_5': avg_wickets_taken_5,
            'avg_fantasy_points_5': avg_fantasy_points_5,
            'matches_played': num_matches,
            'avg_fours': avg_fours,
            'avg_sixes': avg_sixes,
            'avg_catches': avg_catches,
            'avg_runouts': avg_run_outs,
        })

df = pd.DataFrame(temp_df)

In [6]:
df.to_csv('../data/2026_player_match_agg.csv', index=False)

In [9]:
# Get a list of unique matches
match_df = []
for m in unique_matches:
    match_df.append({
        'match_id': m,
        'venue': data[data['match_id'] == m]['venue'].iloc[0],
        'date': data[data['match_id'] == m]['date'].iloc[0],
        'batting_team': data[data['match_id'] == m]['batting_team'].iloc[0],
        'bowling_team': data[data['match_id'] == m]['bowling_team'].iloc[0]
    })

match_df = pd.DataFrame(match_df)
match_df.to_csv('./2026_matchlist.csv', index=False)